# Data Cafe Sales dataset - preprocessing notebook

The objective is to preprocess the `dirty_cafe_sales.csv` dataset by cleaning **'ERROR'**, **'UNKNOWN'** but also missing (`None`) values.

> **Notes:**
> when you created your notebook, you normally already installed the `Numpy` and `Pandas` dependencies
>

### Import dependencies

In [1]:
# use DataPlatform Python SDK 
from forepaas.dwh import connect, update_metas
from forepaas.core.settings import CONFIG
from forepaas.dwh import bulk_insert

# import Python Pandas and Numpy to manage Python dataframe
import pandas as pd
import numpy as np
import os
import requests
import json

### Load datatset from Connectors

Choose your `dirty_cafe_sales` table from **Lakehouse Manager**:

In [ ]:
connector = connect("dwh/default_dataset/")
connector.list()

In [ ]:
df = connector.select("dirty_cafe_sales")

print("\nDataset information:\n", df.info)
print("\nDataset details:\n", df.describe())

### Change datatype for the following attributes

- `item` is a string
- `quantity` is a float
- `price_per_unit` is a float
- `total_spent` is a float

In [ ]:
print("Attributes dtypes:\n", df.dtypes)

In [ ]:
# Columns to clean
columns_to_clean = ['item', 'quantity', 'price_per_unit', 'total_spent']

# Replace 'ERROR' and 'UNKNOWN' with NaN
for col in columns_to_clean:
    print(col)
    df[col] = df[col].replace(['ERROR', 'UNKNOWN', ''], np.nan)
    # Convert numerical columns to float
    if col != 'item': 
        df[col] = df[col].astype(float)

# Check datatypes
print("Attributes dtypes:", df.dtypes)

### Fill in the empty values as much as possible thanks to the correlation between the data

- `item`
- `quantity`
- `price_per_unit`
- `total_spent`

> **⚠️ Warning** - an `item` has a single `price_per_unit` BUT caution, the reverse is not true!

In [6]:
# item / price dictionnary
item_price = {
    'Coffee': 2.0, 'Tea': 1.5, 'Sandwich': 4.0, 'Salad': 3.0,
    'Cake': 3.0, 'Cookie': 1.0, 'Smoothie': 4.0, 'Juice': 3.0
}

# Reverse - Price / item dictionnary
price_item = {price: item for item, price in item_price.items()}

# Define a maximum number of iterations to avoid an infinite loop
max_iterations = 3
iteration = 0

# Loop to fill NaNs as fully as possible thanks to correlation between 'item', 'price_per_unit', 'quantity' and 'total_spent'
while df['item'].notna().sum() > 0 and iteration < max_iterations:

    # total_spent = price_per_unit * quantity
    df['price_per_unit'] = df['price_per_unit'].fillna(df['total_spent'] / df['quantity'])
    df['quantity'] = df['quantity'].fillna(df['total_spent'] / df['price_per_unit'])
    df['total_spent'] = df['total_spent'].fillna(df['price_per_unit'] * df['quantity'])

    # 'Coffee': 2.0, 'Tea': 1.5, 'Sandwich': 4.0, 'Salad': 3.0, 'Cake': 3.0, 'Cookie': 1.0, 'Smoothie': 4.0, 'Juice': 3.0
    df['price_per_unit'] = df['price_per_unit'].fillna(df['item'].map(item_price))
    df['item'] = df['item'].fillna(df['price_per_unit'].map(price_item))

    iteration += 1

In [ ]:
# Display dataset information after replacement
print("Dataset information", df.info())
print("____________________________________________")

### Remove rows where values are still missing

Apply the deletion by looking at the following columns:

- `item`
- `quantity`
- `transaction_date`

In [ ]:
# delete the remaining missing values
df = df.dropna(subset=['item', 'quantity', 'transaction_date'])

# display dataset information after deletion
print("Dataset information", df.info())
print("____________________________________________")

### Save clean dataframe into csv file

You can now save your processed dataframe into a new csv file.

In [ ]:
df.to_csv('clean_cafe_sales.csv', index=False)
df

### [Optional] - Update the clean_cafe_sales table 

You can now connect to the **Lakehouse Manager** and update the table created previously.

In [ ]:
destination = connect("dwh/default_dataset/")
destination.list()

In [12]:
stats = bulk_insert(destination, "clean_cafe_sales", df)

In [ ]:
init_dwh_config()
update_metas()

### Benefit from AI Endpoints for smart data analysis

- Access AI Endpoint access token from envirnoment variables

In [13]:
# Convert dataframe into readable format - JSON
df = pd.read_csv('clean_cafe_sales.csv')
df_analysis = df.drop(['transaction_id'], axis=1)

In [ ]:
df_analysis

- Create **AI Endpoints** request function

A single request is enough to generate all the Python code you need for the dataset analysis!

> Here, you can use the **[Llama 3.3 70B Instruct model](https://llama-3-3-70b-instruct.endpoints.kepler.ai.cloud.ovh.net/doc)** to obtain a Python code that will allow you to analyze your dataset easily.

> The advantage of asking LLM to generate code in Python is that you will be able to reuse it when you add new data to your coffee sales in subsequent months. You will then have the same analysis method!


#### To generate an AI Endpoints API key, acces [https://endpoints.ai.cloud.ovh.net/](https://endpoints.ai.cloud.ovh.net/)

In [15]:
AI_ENDPOINTS_API_KEY = "<your_ai_endpoints_api_key"

In [16]:
def ai_endpoints_llm_data_anaysis(data):
    
    # API url
    url = "https://llama-3-3-70b-instruct.endpoints.kepler.ai.cloud.ovh.net/api/openai_compat/v1/chat/completions"

    # Build prompt message
    message = f"""Generate a Python script to analyze the following café sales data. The script should:
            1. Identify the top 5 best-selling items.
            2. Calculate total revenue.
            3. Find the most common payment method.
            4. Identify seasonal trends in sales.
            5. Visualize the sales trend over time.
            Ensure the script uses pandas, matplotlib, and seaborn.\n
            Data Sample: {data[:500]} \n
            Load data as follow in the code: df = pd.read_csv('clean_cafe_sales.csv')\n
            Return **only the Python code** without any explanations."""
    
    # Define headers and payload
    headers = {
        "Authorization": f"Bearer {AI_ENDPOINTS_API_KEY}",
        "Content-Type": "application/json",
    }
    
    data = {
        "model": "Meta-Llama-3_3-70B-Instruct",
        "messages": [
            {"role": "system", "content": "You are a Python data visualization expert."},
            {"role": "user", "content": message}
        ],
        "temperature": 0,
    }

    # Send request and get answers
    response = requests.post(url, json=data, headers=headers).json()
    content = response['choices'][0]['message']['content']
    
    # format response and print it
    formatted_response = content.split('\n\n')

    with open("data_analysis.py", 'w') as file:
        # Iterate over each section in the formatted response
        for section in formatted_response:
            
            # Filter out lines that start with "```" or "```python"
            filtered_lines = [line for line in section.split('\n') if line.strip() not in ["```", "```python"]]

            # Write the filtered lines to the file
            file.write('\n'.join(filtered_lines) + "\n\n")

-  Ask for **Data Analysis** script

> Then you will be able to reuse it for future Data Analysis if you add new data

In [17]:
# Send your dataset to get generate automatically the Python code for data analysis
csv_data = df_analysis.to_csv(index=False)
data_analysis_answer = ai_endpoints_llm_data_anaysis(csv_data)
print("Data Analysis script in Python has been created: data_analysis.py")

Data Analysis script in Python has been created: data_analysis.py


- Get **Data Analysis** result

You can launch the generated Python code and enjoy the Data Analysis result!

`!python data_analysis.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('clean_cafe_sales.csv')

# Convert transaction_date to datetime
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Identify top 5 best-selling items
top_selling_items = df.groupby('item')['quantity'].sum().sort_values(ascending=False).head(5)
print("Top 5 Best-Selling Items:")
print(top_selling_items)

# Calculate total revenue
total_revenue = df['total_spent'].sum()
print(f"\nTotal Revenue: ${total_revenue:.2f}")

# Find most common payment method
most_common_payment_method = df['payment_method'].mode().values[0]
print(f"\nMost Common Payment Method: {most_common_payment_method}")

# Identify seasonal trends in sales
df['month'] = df['transaction_date'].dt.month
seasonal_trends = df.groupby('month')['total_spent'].sum()
print("\nSeasonal Trends in Sales:")
print(seasonal_trends)

# Visualize sales trend over time
plt.figure(figsize=(10,6))
sns.lineplot(data=df, x='transaction_date', y='total_spent')
plt.title('Sales Trend Over Time')
plt.xlabel('Date')
plt.ylabel('Total Spent')
plt.show()

# Visualize top 5 best-selling items
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='item', order=top_selling_items.index)
plt.title('Top 5 Best-Selling Items')
plt.xlabel('Item')
plt.ylabel('Quantity')
plt.show()

# Visualize payment method distribution
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='payment_method')
plt.title('Payment Method Distribution')
plt.xlabel('Payment Method')
plt.ylabel('Count')
plt.show()

Good job!